# Prokka 基因注释工作流

这个工作流程将：
1. 接受 FNA（核酸序列）文件作为输入
2. 使用 Prokka 进行基因注释和蛋白质预测
3. 输出蛋白质序列供后续结构预测使用

## 系统要求
- JupyterLab/JupyterHub 服务器环境
- 约 5 GB 磁盘空间
- 运行时间取决于序列数量和长度

## 自动安装功能 🆕
- **无需预装 conda/mamba**：Notebook 会自动安装 micromamba
- **自动创建环境**：自动安装 Prokka 及其依赖
- **一键运行**：上传 Notebook 即可在全新服务器上运行

如需禁用自动安装，设置环境变量：`PROTFLOW_AUTO_INSTALL_MICROMAMBA=0`

## 1. 环境检测与设置

In [ ]:
import sys
import os
import subprocess
import shutil
from pathlib import Path

def _str_to_bool(value, default=False):
    if value is None:
        return default
    normalized = str(value).strip().lower()
    if normalized == '':
        return default
    return normalized in {"1", "true", "yes", "y"}

def _load_env_file(env_path: Path):
    if not env_path.exists():
        return
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, val = line.split('=', 1)
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))

PROJECT_ROOT = Path(os.environ.get('PROTFLOW_ROOT', Path.cwd())).resolve()
ENV_FILE = Path(os.environ.get('PROTFLOW_ENV_FILE', PROJECT_ROOT / '.env'))
_load_env_file(ENV_FILE)

IN_COLAB = 'google.colab' in sys.modules
IN_JUPYTERHUB = bool(os.environ.get('JUPYTERHUB_SERVICE_PREFIX'))

if IN_COLAB:
    print("✓ 运行在 Google Colab")
    from google.colab import files, drive
elif IN_JUPYTERHUB:
    print("✓ 运行在 JupyterHub/JupyterLab 服务器环境")
else:
    print("✓ 运行在本地环境")

PROKKA_ENV_NAME = os.environ.get('PROKKA_ENV_NAME', 'prokka')
AUTO_CREATE_PROKKA_ENV = _str_to_bool(os.environ.get('PROTFLOW_AUTO_CREATE_PROKKA', '1'), default=True)

if IN_COLAB:
    WORK_DIR = Path('/content/prokka_workflow')
else:
    default_runs_dir = PROJECT_ROOT / 'prokka_runs'
    WORK_DIR = Path(os.environ.get('PROTFLOW_WORKDIR', default_runs_dir)).expanduser()

WORK_DIR.mkdir(exist_ok=True, parents=True)

if IN_COLAB:
    os.chdir(WORK_DIR)
else:
    print(f"项目根目录: {PROJECT_ROOT}")
    print(f"输出目录: {WORK_DIR.resolve()}")
    if IN_JUPYTERHUB:
        print("提示: 当前处于远程 JupyterLab/Hub，会话结束后输出仍保存在服务器上。")
    else:
        print("提示: 输出内容将写入上述目录，可用终端或文件浏览器访问。")

print(f"使用 PROKKA_ENV_NAME={PROKKA_ENV_NAME}")
print(f"AUTO_CREATE_PROKKA_ENV={'ON' if AUTO_CREATE_PROKKA_ENV else 'OFF'}")
print(f"\n工作目录: {WORK_DIR.resolve()}")

## 2. 安装 Prokka 依赖

本 Notebook 支持自动安装所需的所有依赖：
- **Micromamba**：如果系统中没有 conda/mamba，将自动下载并安装到 `~/.local/bin/`
- **Prokka**：自动创建 conda 环境并安装 Prokka 及其依赖

整个过程自动化，无需手动操作。首次运行约需 5-10 分钟。

In [ ]:
import subprocess
import shutil
import platform
import tempfile
from pathlib import Path
def install_micromamba():
    """
    自动安装 micromamba 到用户目录
    支持 Linux 和 macOS
    """
    print("="*60)
    print("自动安装 Micromamba")
    print("="*60)
    system = platform.system()
    machine = platform.machine()
    # 确定安装目录
    install_dir = Path.home() / '.local' / 'bin'
    install_dir.mkdir(parents=True, exist_ok=True)
    micromamba_bin = install_dir / 'micromamba'
    # 检查是否已经安装
    if micromamba_bin.exists():
        print(f"✅ Micromamba 已存在: {micromamba_bin}")
        if str(install_dir) not in os.environ.get('PATH', ''):
            os.environ['PATH'] = f"{install_dir}:{os.environ.get('PATH', '')}"
        return str(micromamba_bin)
    print(f"\n检测到系统: {system} ({machine})")
    print(f"安装目录: {install_dir}")
    # 确定下载 URL
    if system == 'Linux':
        if machine == 'x86_64':
            url = 'https://micro.mamba.pm/api/micromamba/linux-64/latest'
        elif machine == 'aarch64':
            url = 'https://micro.mamba.pm/api/micromamba/linux-aarch64/latest'
        else:
            raise RuntimeError(f"不支持的 Linux 架构: {machine}")
    elif system == 'Darwin':  # macOS
        if machine == 'arm64':
            url = 'https://micro.mamba.pm/api/micromamba/osx-arm64/latest'
        else:
            url = 'https://micro.mamba.pm/api/micromamba/osx-64/latest'
    else:
        raise RuntimeError(f"不支持的操作系统: {system}")
    print(f"\n📥 正在下载 micromamba...")
    print(f"   URL: {url}")
    try:
        import tarfile
        import urllib.request
        with tempfile.NamedTemporaryFile(suffix='.tar.bz2', delete=False) as tmp_file:
            tmp_path = Path(tmp_file.name)
            urllib.request.urlretrieve(url, tmp_path)
            print(f"✅ 下载完成: {tmp_path.stat().st_size / 1024 / 1024:.2f} MB")
            print("📦 正在解压...")
            with tarfile.open(tmp_path, 'r:bz2') as tar:
                for member in tar.getmembers():
                    if member.name.endswith('bin/micromamba') or member.name == 'bin/micromamba':
                        member.name = 'micromamba'
                        tar.extract(member, install_dir)
                        break
            tmp_path.unlink()
        micromamba_bin.chmod(0o755)
        print(f"✅ Micromamba 安装成功: {micromamba_bin}")
        os.environ['PATH'] = f"{install_dir}:{os.environ.get('PATH', '')}"
        print("\n🔧 正在初始化 micromamba...")
        try:
            subprocess.run([str(micromamba_bin), 'shell', 'init', '-s', 'bash', '-p', str(Path.home() / 'micromamba')],
                          capture_output=True, check=False)
            print("✅ Micromamba 初始化完成")
        except Exception as e:
            print(f"⚠️ 初始化警告（可忽略）: {e}")
        return str(micromamba_bin)
    except Exception as e:
        print(f"\n❌ 安装失败: {e}")
        print("\n请手动安装 micromamba:")
        print("  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
        print("  macOS: brew install micromamba")
        raise
# 检测可用的 conda/mamba 包管理器
CONDA_BIN = None
AUTO_INSTALLED_MICROMAMBA = False
# 首先检查 micromamba 和 mamba (更快)
for cmd in ['micromamba', 'mamba']:
    bin_path = shutil.which(cmd)
    if bin_path:
        CONDA_BIN = (cmd, bin_path)
        print(f"✅ 检测到 {cmd}: {bin_path}")
        break
# 如果没有，检查 conda
if not CONDA_BIN:
    conda_exe = os.environ.get('CONDA_EXE') or shutil.which('conda')
    if conda_exe:
        CONDA_BIN = ('conda', conda_exe)
        print(f"✅ 检测到 conda: {conda_exe}")
# 如果都没有，尝试自动安装 micromamba
if not CONDA_BIN:
    print("\n⚠️ 未检测到 conda/mamba/micromamba")
    auto_install = os.environ.get('PROTFLOW_AUTO_INSTALL_MICROMAMBA', '1')
    if auto_install in ('1', 'true', 'True', 'yes', 'YES'):
        try:
            print("\n🚀 正在自动安装 micromamba...")
            print("   (如不需要，请设置环境变量: PROTFLOW_AUTO_INSTALL_MICROMAMBA=0)")
            micromamba_path = install_micromamba()
            CONDA_BIN = ('micromamba', micromamba_path)
            AUTO_INSTALLED_MICROMAMBA = True
            print(f"\n✅ Micromamba 安装并配置成功！")
            print(f"   路径: {micromamba_path}")
            print(f"   (已添加到当前会话的 PATH)")
        except Exception as e:
            print(f"\n❌ 自动安装失败: {e}")
            print("\n请选择以下任一方式手动安装:")
            print("\n方式 1 - 安装 micromamba (推荐):")
            print("  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
            print("  macOS: brew install micromamba")
            print("\n方式 2 - 安装 conda/mamba:")
            print("  https://docs.conda.io/en/latest/miniconda.html")
            print("\n方式 3 - 设置已安装的 Prokka:")
            print("  export PROKKA_BIN=/path/to/prokka")
            print("="*60)
    else:
        print("\n自动安装已禁用（PROTFLOW_AUTO_INSTALL_MICROMAMBA=0）")
        print("\n请选择以下任一方式手动安装:")
        print("\n方式 1 - 安装 micromamba (推荐):")
        print("  Linux: curl -Ls https://micro.mamba.pm/install.sh | bash")
        print("  macOS: brew install micromamba")
        print("\n方式 2 - 安装 conda/mamba:")
        print("  https://docs.conda.io/en/latest/miniconda.html")
        print("="*60)
def ensure_prokka_available(env_name: str = 'prokka', auto_create: bool = True):
    """检查并确保 Prokka 可用"""
    # 1. 检查环境变量 PROKKA_BIN
    override = os.environ.get('PROKKA_BIN')
    if override:
        prokka_path = Path(override).expanduser()
        if not prokka_path.exists():
            raise FileNotFoundError(f"PROKKA_BIN 指向的文件不存在: {prokka_path}")
        print(f"✅ 使用 PROKKA_BIN: {prokka_path}")
        return [str(prokka_path)]
    # 2. 检查是否有 conda/mamba 环境
    if CONDA_BIN:
        conda_cmd, conda_path = CONDA_BIN
        # 检查环境是否存在
        try:
            env_list = subprocess.run(
                [conda_path, 'env', 'list'],
                capture_output=True,
                text=True,
                check=True
            )
            env_exists = env_name in env_list.stdout
        except Exception:
            env_exists = False
        # 构建 prokka 命令
        if conda_cmd == 'conda':
            prokka_cmd = ['conda', 'run', '-n', env_name, 'prokka']
        else:
            prokka_cmd = [conda_path, 'run', '-n', env_name, 'prokka']
        # 如果环境不存在，尝试创建
        if not env_exists:
            if not auto_create:
                raise RuntimeError(
                    f"未检测到 {conda_cmd} 环境 '{env_name}'，并且自动创建被禁用。\n"
                    f"请手动创建: {conda_cmd} create -n {env_name} -c conda-forge -c bioconda prokka"
                )
            print(f"📦 正在使用 {conda_cmd} 创建环境: {env_name}")
            print("   这可能需要 5-10 分钟...\n")
            try:
                create_cmd = [
                    conda_path, 'create', '-y', '-n', env_name,
                    '-c', 'conda-forge', '-c', 'bioconda', '-c', 'defaults',
                    'prokka'
                ]
                subprocess.run(create_cmd, check=True)
                print('✅ Prokka 环境创建成功！')
            except subprocess.CalledProcessError as e:
                raise RuntimeError(
                    f"创建 Prokka 环境失败。\n"
                    f"请手动安装: {conda_cmd} create -n {env_name} -c conda-forge -c bioconda prokka"
                ) from e
        else:
            print(f"✅ 环境 '{env_name}' 已存在")
        # 验证 Prokka 可用性
        try:
            version_cmd = prokka_cmd + ['--version']
            result = subprocess.run(version_cmd, capture_output=True, text=True, check=True)
            print(f"✅ Prokka 可用: {result.stdout.strip()}")
            return prokka_cmd
        except subprocess.CalledProcessError as e:
            print(f"❌ Prokka 验证失败: {e}")
            raise
    else:
        raise RuntimeError(
            "未找到可用的 Prokka 安装。\n"
            "请安装 conda/mamba/micromamba 或设置 PROKKA_BIN 环境变量���"
        )
# 确保 Prokka 可用
try:
    PROKKA_CMD = ensure_prokka_available(PROKKA_ENV_NAME, AUTO_CREATE_PROKKA_ENV)
    print(f"\n✅ Prokka 环境配置完成")
    print(f"   命令: {' '.join(PROKKA_CMD)}")
except Exception as e:
    print(f"\n❌ Prokka 环境配置失败: {e}")
    raise

## 3. 安装 Python 包依赖

安装工作流所需的 Python 包（BioPython 等）

In [ ]:
import sys
import subprocess

packages = ['biopython', 'tqdm', 'ipywidgets']
print("正在安装 Python 包...")
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f"  ✓ {pkg}")
    except subprocess.CalledProcessError as e:
        print(f"  ⚠ {pkg} 安装失败: {e}")

print("\n✅ Python 包安装完成")

## 4. 导入必要的库

In [ ]:
from Bio import SeqIO
from pathlib import Path
from tqdm.auto import tqdm
import subprocess
import json

print("✓ 所有库加载成功")

## 5. 上传输入文件

上传你的 FNA（核酸序列）文件

In [ ]:
if IN_COLAB:
    # Colab: 上传文件
    print("请上传你的 FNA 文件：")
    uploaded = files.upload()
    
    if not uploaded:
        raise ValueError("未上传任何文件")
    
    uploaded_file = list(uploaded.keys())[0]
    INPUT_FNA = WORK_DIR / uploaded_file
    
    import shutil
    shutil.move(uploaded_file, INPUT_FNA)
    
    print(f"\n✓ 文件已上传: {INPUT_FNA}")
    print(f"  大小: {INPUT_FNA.stat().st_size / 1024:.1f} KB")
else:
    # 本地/JupyterHub: 指定文件路径
    INPUT_FNA = WORK_DIR / 'input.fna'
    
    if not INPUT_FNA.exists():
        print(f"⚠️ 文件不存在: {INPUT_FNA}")
        print(f"\n请将你的 FNA 文件放置在: {WORK_DIR}")
        print(f"或修改上面的代码设置正确的文件路径")
    else:
        print(f"✓ 输入文件: {INPUT_FNA}")
        print(f"  大小: {INPUT_FNA.stat().st_size / 1024:.1f} KB")

## 6. 配置 Prokka 参数

In [ ]:
# Prokka 配置
RUN_PREFIX = "prokka_output"
KINGDOM = "Bacteria"
CPUS = 4

PROKKA_OUTPUT_DIR = WORK_DIR / RUN_PREFIX
PROKKA_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"配置:")
print(f"  输入文件: {INPUT_FNA}")
print(f"  输出目录: {PROKKA_OUTPUT_DIR}")
print(f"  前缀: {RUN_PREFIX}")
print(f"  Kingdom: {KINGDOM}")
print(f"  CPUs: {CPUS}")

## 7. 运行 Prokka 基因注释

In [ ]:
print(f"\n{'='*60}")
print("运行 Prokka 基因注释")
print(f"{'='*60}")

# 构建 Prokka 命令
cmd = PROKKA_CMD + [
    "--outdir", str(PROKKA_OUTPUT_DIR),
    "--prefix", RUN_PREFIX,
    "--kingdom", KINGDOM,
    "--cpus", str(CPUS),
    "--force",  # 覆盖已存在的输出
    str(INPUT_FNA)
]

print(f"运行命令: {' '.join(cmd)}")
print("\n正在运行 Prokka（这可能需要几分钟）...\n")

try:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print("\n✓ Prokka 运行成功！")

    # 显示统计信息
    stats_file = PROKKA_OUTPUT_DIR / f"{RUN_PREFIX}.txt"
    if stats_file.exists():
        print("\n注释统计:")
        print(stats_file.read_text())
except subprocess.CalledProcessError as e:
    print(f"\n✗ Prokka 运行失败: {e}")
    print(f"错误输出: {e.stderr}")
    raise

## 8. 分析 Prokka 输出

In [ ]:
# 读取蛋白质序列
faa_file = PROKKA_OUTPUT_DIR / f"{RUN_PREFIX}.faa"

if not faa_file.exists():
    print(f"⚠️ 未找到蛋白质文件: {faa_file}")
else:
    proteins = list(SeqIO.parse(faa_file, "fasta"))
    
    print(f"\n{'='*60}")
    print("Prokka 结果摘要")
    print(f"{'='*60}")
    print(f"总蛋白质数: {len(proteins)}")
    
    lengths = [len(p.seq) for p in proteins]
    if lengths:
        print(f"\n序列长度统计:")
        print(f"  最短: {min(lengths)} aa")
        print(f"  最长: {max(lengths)} aa")
        print(f"  平均: {sum(lengths)/len(lengths):.1f} aa")

## 9. 查看输出文件

In [ ]:
print(f"\n{'='*60}")
print("Prokka 输出文件")
print(f"{'='*60}")
print(f"\n输出目录: {PROKKA_OUTPUT_DIR}\n")

output_files = sorted(PROKKA_OUTPUT_DIR.glob("*"))
for f in output_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:30s} ({size_kb:>8.1f} KB)")

## 10. 下载结果（Colab 用户）

In [ ]:
if IN_COLAB:
    print("下载结果文件:")
    key_files = [f"{RUN_PREFIX}.faa", f"{RUN_PREFIX}.gbk", f"{RUN_PREFIX}.gff"]
    
    for filename in key_files:
        filepath = PROKKA_OUTPUT_DIR / filename
        if filepath.exists():
            files.download(str(filepath))
            print(f"  ✓ {filename}")
else:
    print("结果已保存在服务器上:")
    print(f"  {PROKKA_OUTPUT_DIR.resolve()}")